In [2]:
from pathlib import Path
import os
import re
import csv
import json
import pandas as pd
from datetime import datetime

print("Python environment is ready.")

Python environment is ready.


In [3]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\DS Project\credit-portfolio-risk-modeling"
)

DATA_LOCATIONS = {
    "2006_2010": PROJECT_ROOT / "dataset",
    "2011_2012": Path(
        r"E:\DS_project_backup\credit-risk-portfolio\data"
    )
}

for name, path in DATA_LOCATIONS.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")
    print("-" * 60)

2006_2010: D:\DS Project\credit-portfolio-risk-modeling\dataset
Exists: True
------------------------------------------------------------
2011_2012: E:\DS_project_backup\credit-risk-portfolio\data
Exists: True
------------------------------------------------------------


In [4]:
for name, folder in DATA_LOCATIONS.items():

    files = list(folder.glob("*.txt"))

    print(f"\n{name}")
    print(f"Number of TXT files: {len(files)}")


2006_2010
Number of TXT files: 40

2011_2012
Number of TXT files: 16


In [5]:
records = []

pattern = re.compile(
    r"^(orig|perf)_(\d{4})Q([1-4])\.txt$",
    re.IGNORECASE
)

for location_name, folder in DATA_LOCATIONS.items():

    for file_path in folder.glob("*.txt"):

        match = pattern.match(file_path.name)

        if not match:
            print("Unrecognized filename:", file_path.name)
            continue

        file_type = match.group(1).lower()
        year = int(match.group(2))
        quarter = int(match.group(3))

        records.append({
            "file_name": file_path.name,
            "file_type": file_type,
            "year": year,
            "quarter": quarter,
            "vintage": f"{year}Q{quarter}",
            "location": location_name,
            "full_path": str(file_path),
            "size_bytes": file_path.stat().st_size,
            "size_mb": round(
                file_path.stat().st_size / (1024 ** 2), 2
            ),
            "modified_time": datetime.fromtimestamp(
                file_path.stat().st_mtime
            )
        })

inventory = pd.DataFrame(records)

inventory = inventory.sort_values(
    ["year", "quarter", "file_type"]
).reset_index(drop=True)

inventory.head(10)

,file_name,file_type,year,quarter,vintage,location,full_path,size_bytes,size_mb,modified_time
0,orig_2006Q1.txt,orig,2006,1,2006Q1,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,38231376,36.46,2026-11-09 16:23:38.709747
1,perf_2006Q1.txt,perf,2006,1,2006Q1,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,2391330816,2280.55,2026-11-09 16:24:42.043555
2,orig_2006Q2.txt,orig,2006,2,2006Q2,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,39446114,37.62,2026-11-09 16:26:04.154723
3,perf_2006Q2.txt,perf,2006,2,2006Q2,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,2221270439,2118.37,2026-11-09 16:26:52.167957
4,orig_2006Q3.txt,orig,2006,3,2006Q3,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,36432889,34.75,2026-11-09 16:27:11.843308
5,perf_2006Q3.txt,perf,2006,3,2006Q3,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,1972451970,1881.08,2026-11-09 16:27:50.220225
6,orig_2006Q4.txt,orig,2006,4,2006Q4,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,38919124,37.12,2026-11-09 16:31:09.189398
7,perf_2006Q4.txt,perf,2006,4,2006Q4,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,2272059777,2166.81,2026-11-09 16:31:52.842677
8,orig_2007Q1.txt,orig,2007,1,2007Q1,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,38848215,37.05,2026-11-09 16:36:23.460560
9,perf_2007Q1.txt,perf,2007,1,2007Q1,2006_2010,D:\DS Project\credit-portfolio-risk-modeling\d...,2254628040,2150.18,2026-11-09 16:37:11.917092


In [6]:
summary = (
    inventory
    .groupby(["year", "file_type"])
    .agg(
        file_count=("file_name", "count"),
        total_size_mb=("size_mb", "sum")
    )
    .reset_index()
)

summary["total_size_gb"] = (
    summary["total_size_mb"] / 1024
).round(2)

summary

,year,file_type,file_count,total_size_mb,total_size_gb
0,2006,orig,4,145.95,0.14
1,2006,perf,4,8446.81,8.25
2,2007,orig,4,150.96,0.15
3,2007,perf,4,8218.37,8.03
4,2008,orig,4,149.86,0.15
5,2008,perf,4,6876.49,6.72
6,2009,orig,4,269.37,0.26
7,2009,perf,4,15161.54,14.81
8,2010,orig,4,224.00,0.22
9,2010,perf,4,13742.33,13.42


In [7]:
inventory.groupby("file_type").agg(
    number_of_files=("file_name", "count"),
    total_size_gb=(
        "size_mb",
        lambda x: round(x.sum() / 1024, 2)
    ),
    average_size_mb=("size_mb", "mean"),
    largest_file_mb=("size_mb", "max")
)

,number_of_files,total_size_gb,average_size_mb,largest_file_mb
file_type,,,,
orig,28,1.33,48.661786,86.35
perf,28,81.44,2978.325357,6090.72


In [10]:
expected_vintages = [
    f"{year}Q{quarter}"
    for year in range(2006, 2013)
    for quarter in range(1, 5)
]

available = set(
    zip(inventory["vintage"], inventory["file_type"])
)

pair_records = []

for vintage in expected_vintages:

    has_orig = (vintage, "orig") in available
    has_perf = (vintage, "perf") in available

    pair_records.append({
        "vintage": vintage,
        "origination_available": has_orig,
        "performance_available": has_perf,
        "complete_pair": has_orig and has_perf
    })

pair_check = pd.DataFrame(pair_records)

pair_check

,vintage,origination_available,performance_available,complete_pair
0,2006Q1,True,True,True
1,2006Q2,True,True,True
2,2006Q3,True,True,True
3,2006Q4,True,True,True
4,2007Q1,True,True,True
5,2007Q2,True,True,True
6,2007Q3,True,True,True
7,2007Q4,True,True,True
8,2008Q1,True,True,True
9,2008Q2,True,True,True


In [11]:
print("Expected quarters:", len(expected_vintages))
print("Complete pairs:", pair_check["complete_pair"].sum())

print("\nMissing Origination:")
print(
    pair_check.loc[
        ~pair_check["origination_available"],
        "vintage"
    ].tolist()
)

print("\nMissing Performance:")
print(
    pair_check.loc[
        ~pair_check["performance_available"],
        "vintage"
    ].tolist()
)

Expected quarters: 28
Complete pairs: 28

Missing Origination:
[]

Missing Performance:
[]


In [12]:
sample_vintage = "2012Q4"

sample_files = inventory[
    inventory["vintage"] == sample_vintage
]

sample_files[
    ["file_name", "file_type", "full_path", "size_mb"]
]

,file_name,file_type,full_path,size_mb
54,orig_2012Q4.txt,orig,E:\DS_project_backup\credit-risk-portfolio\dat...,72.29
55,perf_2012Q4.txt,perf,E:\DS_project_backup\credit-risk-portfolio\dat...,6090.72


In [13]:
def inspect_first_lines(file_path, n=5):

    print(f"\nFILE: {file_path.name}")
    print("=" * 80)

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as f:

        for i in range(n):
            line = f.readline()

            if not line:
                break

            print(f"\nLine {i + 1}:")
            print(repr(line[:500]))

for _, row in sample_files.iterrows():

    inspect_first_lines(
        Path(row["full_path"]),
        n=5
    )


FILE: orig_2012Q4.txt

Line 1:
'804|201301|N|202712||0|1|P|71|18|156000|71|2.5|R|N|FRM|IA|SF|513|F12Q40000001|N|180|2|OTHER|N|||N|7|N|9999\n'

Line 2:
'715|201302|N|204301|37900|0|1|P|71|26|53000|71|4|R|N|FRM|IL|SF|615|F12Q40000002|C|360|2|OTHER|N|||N|7|N|9999\n'

Line 3:
'710|201212|N|204211|33340|0|1|P|80|45|216000|80|3.125|R|N|FRM|WI|SF|530|F12Q40000003|C|360|2|OTHER|N|||N|7|N|9999\n'

Line 4:
'776|201212|N|204211|31740|30|1|P|95|31|140000|95|3.375|R|N|FRM|KS|SF|665|F12Q40000004|N|360|2|OTHER|N|||N|7|N|9999\n'

Line 5:
'755|201212|N|204211|16974|0|1|P|106|999|339000|106|3.5|R|N|FRM|IL|SF|605|F12Q40000005|N|360|1|OTHER|N|F09Q20008428||Y|7|N|9999\n'

FILE: perf_2012Q4.txt

Line 1:
'F12Q40000001|201212|156000.00|00|0|180|||||2.500|0.00|||||||||||0.00|||999||||||156000.00|7|OTHER|\n'

Line 2:
'F12Q40000001|201301|155000.00|00|1|179|||||2.500|0.00|||||||||||0.00|||999||||||155000.00|7|OTHER|\n'

Line 3:
'F12Q40000001|201302|155000.00|00|2|178|||||2.500|0.00|||||||||||0.00|||999||||||155

In [17]:
def inspect_field_counts(file_path, n=1000):

    field_counts = {}

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as f:

        reader = csv.reader(f, delimiter="|")

        for i, row in enumerate(reader):

            if i >= n:
                break

            count = len(row)

            field_counts[count] = (
                field_counts.get(count, 0) + 1
            )

    return field_counts


for _, row in sample_files.iterrows():

    file_path = Path(row["full_path"])

    counts = inspect_field_counts(
        file_path,
        n=10_000
    )

    print(f"\n{file_path.name}")
    print("Field count distribution:", counts)


orig_2012Q4.txt
Field count distribution: {31: 10000}

perf_2012Q4.txt
Field count distribution: {35: 10000}


In [18]:
def inspect_first_row(file_path):

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as f:

        first_line = f.readline().strip()

    fields = first_line.split("|")

    return {
        "file": file_path.name,
        "first_row": first_line[:300],
        "number_of_fields": len(fields),
        "looks_like_header": any(
            word in first_line.lower()
            for word in [
                "loan_sequence_number",
                "credit_score",
                "monthly_reporting"
            ]
        )
    }


for _, row in sample_files.iterrows():

    result = inspect_first_row(
        Path(row["full_path"])
    )

    print(result)

{'file': 'orig_2012Q4.txt', 'first_row': '804|201301|N|202712||0|1|P|71|18|156000|71|2.5|R|N|FRM|IA|SF|513|F12Q40000001|N|180|2|OTHER|N|||N|7|N|9999', 'number_of_fields': 31, 'looks_like_header': False}
{'file': 'perf_2012Q4.txt', 'first_row': 'F12Q40000001|201212|156000.00|00|0|180|||||2.500|0.00|||||||||||0.00|||999||||||156000.00|7|OTHER|', 'number_of_fields': 35, 'looks_like_header': False}


In [20]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_audit"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

inventory.to_csv(
    OUTPUT_DIR / "file_inventory.csv",
    index=False,
    encoding="utf-8-sig"
)

pair_check.to_csv(
    OUTPUT_DIR / "quarter_pair_check.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    OUTPUT_DIR / "annual_file_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Audit outputs saved to:", OUTPUT_DIR)

Audit outputs saved to: D:\DS Project\credit-portfolio-risk-modeling\outputs\data_audit


In [21]:
orig_path = Path(
    sample_files.loc[
        sample_files["file_type"] == "orig",
        "full_path"
    ].iloc[0]
)

perf_path = Path(
    sample_files.loc[
        sample_files["file_type"] == "perf",
        "full_path"
    ].iloc[0]
)

# Read only a small sample
orig_sample = pd.read_csv(
    orig_path,
    sep="|",
    header=None,
    nrows=1000,
    dtype=str
)

perf_sample = pd.read_csv(
    perf_path,
    sep="|",
    header=None,
    nrows=1000,
    dtype=str
)

# Origination: column 20 (1-based) => index 19
orig_ids = set(orig_sample.iloc[:, 19].dropna())

# Performance: column 1 => index 0
perf_ids = set(perf_sample.iloc[:, 0].dropna())

print("Origination sample IDs:", len(orig_ids))
print("Performance sample IDs:", len(perf_ids))
print("Common IDs:", len(orig_ids & perf_ids))
print("Orig IDs not in Performance sample:",
      len(orig_ids - perf_ids))

Origination sample IDs: 1000
Performance sample IDs: 8
Common IDs: 8
Orig IDs not in Performance sample: 992


In [22]:
print(
    "Origination duplicate IDs:",
    orig_sample.iloc[:, 19].duplicated().sum()
)

print(
    "Performance duplicate loan-month rows:",
    perf_sample.duplicated(
        subset=[0, 1]
    ).sum()
)

Origination duplicate IDs: 0
Performance duplicate loan-month rows: 0


In [23]:
# Count observations per loan in the sample

loan_frequency = (
    perf_sample
    .groupby(0)
    .size()
    .reset_index(name="number_of_months")
    .rename(columns={0: "loan_id"})
    .sort_values("number_of_months", ascending=False)
)

print("Unique loans:", len(loan_frequency))

display(loan_frequency.head(20))

Unique loans: 8


,loan_id,number_of_months
3,F12Q40000004,164
4,F12Q40000005,164
0,F12Q40000001,163
6,F12Q40000007,162
5,F12Q40000006,119
2,F12Q40000003,100
1,F12Q40000002,91
7,F12Q40000008,37


In [24]:
# Select one loan
selected_loan = "F12Q40000001"

loan_history = (
    perf_sample.loc[
        perf_sample[0] == selected_loan,
        [0, 1, 2, 3, 4, 5]
    ]
    .copy()
)

loan_history.columns = [
    "loan_id",
    "reporting_period",
    "current_actual_upb",
    "delinquency_status",
    "loan_age",
    "remaining_months"
]

display(loan_history)

,loan_id,reporting_period,current_actual_upb,delinquency_status,loan_age,remaining_months
0,F12Q40000001,201212,156000.00,00,0,180
1,F12Q40000001,201301,155000.00,00,1,179
2,F12Q40000001,201302,155000.00,00,2,178
3,F12Q40000001,201303,154000.00,00,3,177
4,F12Q40000001,201304,153000.00,00,4,176
...,...,...,...,...,...,...
158,F12Q40000001,202602,22345.15,00,158,22
159,F12Q40000001,202603,21351.51,00,159,21
160,F12Q40000001,202604,20355.80,00,160,20
161,F12Q40000001,202605,19358.02,00,161,19


In [25]:
periods = loan_history["reporting_period"].astype(str)

print("First reporting period:", periods.iloc[0])
print("Last reporting period:", periods.iloc[-1])

print(
    "Periods sorted:",
    periods.is_monotonic_increasing
)

print(
    "Duplicate reporting periods:",
    periods.duplicated().sum()
)

First reporting period: 201212
Last reporting period: 202606
Periods sorted: True
Duplicate reporting periods: 0
